# Farm-to-Fork Cold-Chain Network Design & Stochastic Supply Planning
**Course:** Supply Chain Analytics (Term 5, PGDM-BDA 2026–27) — Goa Institute of Management  
**Sector:** Agriculture (Western India Horticulture Corridor: Maharashtra – Goa – Karnataka)  
**Primary Methods:** Goal Programming Facility Location & Two-Stage Stochastic Optimization (CLO 5), SARIMA/Seasonal-AR Forecasting (CLO 4), Descriptive KPI Analytics (CLO 3)

---
### Quick-Start Instructions (Local Hosting)
1. **Run All Cells** (`Kernel -> Restart & Run All` or `Run All` in Jupyter/VS Code).
2. Cell 2 checks/installs required Python packages (`streamlit`, `plotly`, `statsmodels`, `pulp`, `pandas`, `numpy`, `scipy`).
3. Cell 3 generates the **built-in 104-week sample dataset**, **72-arc network matrix**, and **blank input template** locally.
4. Cell 4 runs standalone **Model Validation & Test Cases** directly in the notebook output.
5. Cell 5 writes the complete multi-tab web application (`app.py`).
6. Cell 6 launches the **locally hosted web app** at `http://localhost:8501`.


In [ ]:
# CELL 1: Environment Check & Auto-Install Dependencies
import importlib
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "scipy": "scipy",
    "statsmodels": "statsmodels",
    "pulp": "pulp",
    "plotly": "plotly",
    "streamlit": "streamlit",
}

missing = []
for mod_name, pip_name in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(mod_name)
    except ImportError:
        missing.append(pip_name)

if missing:
    print(f"Installing missing packages: {missing} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("All required packages installed successfully.")
else:
    print("All required packages are already installed.")


In [ ]:
# CELL 2: Self-Contained Sample Dataset & Blank Input Template Generator
import datetime
import math
import numpy as np
import pandas as pd

np.random.seed(42)

CLUSTERS = [
    {"id": "FC01", "name": "Nashik (Pimpalgaon)", "state": "Maharashtra", "base_weekly_mt": 420.0, "season_amp": 0.32, "peak_week": 8},
    {"id": "FC02", "name": "Narayangaon (Junner)", "state": "Maharashtra", "base_weekly_mt": 360.0, "season_amp": 0.35, "peak_week": 12},
    {"id": "FC03", "name": "Satara (Wai-Karad)", "state": "Maharashtra", "base_weekly_mt": 250.0, "season_amp": 0.28, "peak_week": 16},
    {"id": "FC04", "name": "Sangli (Miraj-Tasgaon)", "state": "Maharashtra", "base_weekly_mt": 290.0, "season_amp": 0.30, "peak_week": 10},
    {"id": "FC05", "name": "Kolhapur (Shirol)", "state": "Maharashtra", "base_weekly_mt": 220.0, "season_amp": 0.25, "peak_week": 18},
    {"id": "FC06", "name": "Belagavi (Gokak)", "state": "Karnataka", "base_weekly_mt": 340.0, "season_amp": 0.33, "peak_week": 42},
    {"id": "FC07", "name": "Dharwad (Hubballi)", "state": "Karnataka", "base_weekly_mt": 260.0, "season_amp": 0.31, "peak_week": 44},
    {"id": "FC08", "name": "Kolar (Mulbagal)", "state": "Karnataka", "base_weekly_mt": 460.0, "season_amp": 0.38, "peak_week": 38},
]

PACKHOUSES = [
    {"id": "PH01", "name": "Ozarkhed Packhouse Hub (Nashik)", "corridor": "North MH NH60", "weekly_capacity_mt": 850.0, "annualized_capex_lakhs": 68.0, "weekly_fixed_opex_inr": 145000.0, "precool_var_cost_inr_mt": 620.0, "precool_loss_reduction_factor": 0.68},
    {"id": "PH02", "name": "Chakan-Narayangaon Agro Hub (Pune N)", "corridor": "Central MH NH60", "weekly_capacity_mt": 750.0, "annualized_capex_lakhs": 64.0, "weekly_fixed_opex_inr": 135000.0, "precool_var_cost_inr_mt": 590.0, "precool_loss_reduction_factor": 0.66},
    {"id": "PH03", "name": "Karad-Satara Integrated Packhouse", "corridor": "West MH NH48", "weekly_capacity_mt": 650.0, "annualized_capex_lakhs": 54.0, "weekly_fixed_opex_inr": 115000.0, "precool_var_cost_inr_mt": 560.0, "precool_loss_reduction_factor": 0.65},
    {"id": "PH04", "name": "Kagal-Kolhapur Cold Chain Park", "corridor": "South MH-Konkan", "weekly_capacity_mt": 600.0, "annualized_capex_lakhs": 50.0, "weekly_fixed_opex_inr": 108000.0, "precool_var_cost_inr_mt": 550.0, "precool_loss_reduction_factor": 0.64},
    {"id": "PH05", "name": "Belagavi-Kittur Border Logistics Hub", "corridor": "NW KA-Goa NH48", "weekly_capacity_mt": 700.0, "annualized_capex_lakhs": 56.0, "weekly_fixed_opex_inr": 118000.0, "precool_var_cost_inr_mt": 540.0, "precool_loss_reduction_factor": 0.67},
    {"id": "PH06", "name": "Hoskote-Kolar Mega Packhouse", "corridor": "East KA NH75", "weekly_capacity_mt": 800.0, "annualized_capex_lakhs": 66.0, "weekly_fixed_opex_inr": 140000.0, "precool_var_cost_inr_mt": 600.0, "precool_loss_reduction_factor": 0.68},
]

DEMAND_HUBS = [
    {"id": "DH01", "name": "Mumbai Metro DC (Vashi)", "state": "Maharashtra", "base_weekly_demand_mt": 820.0, "target_fill_rate": 0.96, "spot_cost_inr_mt": 31500.0},
    {"id": "DH02", "name": "Pune Metro DC (Marketyard)", "state": "Maharashtra", "base_weekly_demand_mt": 540.0, "target_fill_rate": 0.95, "spot_cost_inr_mt": 29500.0},
    {"id": "DH03", "name": "Goa Coastal Hub (Panaji/Margao)", "state": "Goa", "base_weekly_demand_mt": 260.0, "target_fill_rate": 0.95, "spot_cost_inr_mt": 33800.0},
    {"id": "DH04", "name": "Bengaluru Metro DC (Yeshwanthpur)", "state": "Karnataka", "base_weekly_demand_mt": 760.0, "target_fill_rate": 0.96, "spot_cost_inr_mt": 30500.0},
]

FC_TO_PH_DIST = {
    ("FC01", "PH01"): 28,  ("FC01", "PH02"): 165, ("FC01", "PH03"): 330, ("FC01", "PH04"): 445, ("FC01", "PH05"): 550, ("FC01", "PH06"): 980,
    ("FC02", "PH01"): 140, ("FC02", "PH02"): 35,  ("FC02", "PH03"): 195, ("FC02", "PH04"): 310, ("FC02", "PH05"): 415, ("FC02", "PH06"): 855,
    ("FC03", "PH01"): 320, ("FC03", "PH02"): 145, ("FC03", "PH03"): 32,  ("FC03", "PH04"): 125, ("FC03", "PH05"): 235, ("FC03", "PH06"): 680,
    ("FC04", "PH01"): 390, ("FC04", "PH02"): 230, ("FC04", "PH03"): 75,  ("FC04", "PH04"): 55,  ("FC04", "PH05"): 155, ("FC04", "PH06"): 615,
    ("FC05", "PH01"): 435, ("FC05", "PH02"): 265, ("FC05", "PH03"): 115, ("FC05", "PH04"): 24,  ("FC05", "PH05"): 115, ("FC05", "PH06"): 590,
    ("FC06", "PH01"): 540, ("FC06", "PH02"): 370, ("FC06", "PH03"): 225, ("FC06", "PH04"): 110, ("FC06", "PH05"): 26,  ("FC06", "PH06"): 495,
    ("FC07", "PH01"): 615, ("FC07", "PH02"): 445, ("FC07", "PH03"): 300, ("FC07", "PH04"): 185, ("FC07", "PH05"): 78,  ("FC07", "PH06"): 415,
    ("FC08", "PH01"): 1010,("FC08", "PH02"): 875, ("FC08", "PH03"): 715, ("FC08", "PH04"): 615, ("FC08", "PH05"): 520, ("FC08", "PH06"): 38,
}

PH_TO_DH_DIST = {
    ("PH01", "DH01"): 175, ("PH01", "DH02"): 210, ("PH01", "DH03"): 660, ("PH01", "DH04"): 1020,
    ("PH02", "DH01"): 155, ("PH02", "DH02"): 42,  ("PH02", "DH03"): 490, ("PH02", "DH04"): 860,
    ("PH03", "DH01"): 290, ("PH03", "DH02"): 150, ("PH03", "DH03"): 335, ("PH03", "DH04"): 695,
    ("PH04", "DH01"): 385, ("PH04", "DH02"): 240, ("PH04", "DH03"): 215, ("PH04", "DH04"): 595,
    ("PH05", "DH01"): 490, ("PH05", "DH02"): 345, ("PH05", "DH03"): 145, ("PH05", "DH04"): 485,
    ("PH06", "DH01"): 995, ("PH06", "DH02"): 855, ("PH06", "DH03"): 615, ("PH06", "DH04"): 45,
}

start_date = datetime.date(2024, 11, 4)
weekly_rows = []
for w in range(1, 105):
    curr_date = start_date + datetime.timedelta(weeks=w - 1)
    cal_week = curr_date.isocalendar()[1]
    month = curr_date.month
    if month in (11, 12, 1, 2):
        season = "Rabi (Winter)"
        ambient_temp_c = 27.5 + 2.5 * math.sin(2 * math.pi * (w - 6) / 52.0) + np.random.normal(0, 0.9)
        rel_humidity_pct = 55.0 + np.random.normal(0, 4.0)
    elif month in (3, 4, 5):
        season = "Zaid (Summer)"
        ambient_temp_c = 36.5 + 3.8 * math.sin(2 * math.pi * (w - 18) / 52.0) + np.random.normal(0, 1.1)
        rel_humidity_pct = 42.0 + np.random.normal(0, 3.5)
    else:
        season = "Kharif (Monsoon)"
        ambient_temp_c = 29.8 + 2.0 * math.cos(2 * math.pi * (w - 32) / 52.0) + np.random.normal(0, 1.0)
        rel_humidity_pct = 81.0 + np.random.normal(0, 4.5)

    ambient_temp_c = round(max(22.0, min(43.5, ambient_temp_c)), 1)
    rel_humidity_pct = round(max(30.0, min(95.0, rel_humidity_pct)), 1)
    trend_factor = 1.0 + 0.00085 * w
    weather_shock = "HEATWAVE" if ambient_temp_c > 38.2 else ("HEAVY_RAIN" if rel_humidity_pct > 87.5 else "NORMAL")
    shock_yield = 0.84 if weather_shock != "NORMAL" else 1.00

    c_sup = {}
    for c in CLUSTERS:
        wave = 1.0 + c["season_amp"] * math.cos(2 * math.pi * (cal_week - c["peak_week"]) / 52.0)
        c_sup[c["id"]] = max(80.0, round(c["base_weekly_mt"] * trend_factor * wave * shock_yield * np.random.normal(1.0, 0.055), 1))
    tot_sup = round(sum(c_sup.values()), 1)

    h_dem = {}
    for d in DEMAND_HUBS:
        d_seas = 1.0 + (0.22 * math.cos(2 * math.pi * (cal_week - 2) / 52.0) if d["id"] == "DH03" else 0.08 * math.sin(2 * math.pi * (cal_week - 40) / 52.0))
        h_dem[d["id"]] = round(d["base_weekly_demand_mt"] * trend_factor * d_seas * np.random.normal(1.0, 0.035), 1)
    tot_dem = round(sum(h_dem.values()), 1)

    amb_loss = round(0.085 + 0.0028 * max(0.0, ambient_temp_c - 25.0) + (0.018 if weather_shock != "NORMAL" else 0.0), 4)
    cold_loss = round(0.028 + 0.0005 * max(0.0, ambient_temp_c - 25.0) + (0.004 if weather_shock != "NORMAL" else 0.0), 4)
    price = round(max(9.5, min(68.0, 26.5 * ((tot_sup / max(1.0, tot_dem)) ** -1.35) * np.random.normal(1.0, 0.04))), 2)

    weekly_rows.append({
        "Week_Index": w,
        "Week_Start_Date": curr_date.strftime("%Y-%m-%d"),
        "ISO_Year_Week": f"{curr_date.year}-W{cal_week:02d}",
        "Agri_Season": season,
        "Ambient_Temp_C": ambient_temp_c,
        "Rel_Humidity_Pct": rel_humidity_pct,
        "Weather_Regime": weather_shock,
        "FC01_Nashik_MT": c_sup["FC01"],
        "FC02_Narayangaon_MT": c_sup["FC02"],
        "FC03_Satara_MT": c_sup["FC03"],
        "FC04_Sangli_MT": c_sup["FC04"],
        "FC05_Kolhapur_MT": c_sup["FC05"],
        "FC06_Belagavi_MT": c_sup["FC06"],
        "FC07_Dharwad_MT": c_sup["FC07"],
        "FC08_Kolar_MT": c_sup["FC08"],
        "Total_Cluster_Supply_MT": tot_sup,
        "DH01_Mumbai_Demand_MT": h_dem["DH01"],
        "DH02_Pune_Demand_MT": h_dem["DH02"],
        "DH03_Goa_Demand_MT": h_dem["DH03"],
        "DH04_Bengaluru_Demand_MT": h_dem["DH04"],
        "Total_Hub_Demand_MT": tot_dem,
        "Ambient_Spoilage_Pct": amb_loss,
        "ColdChain_Spoilage_Pct": cold_loss,
        "Mandi_Modal_Price_INR_Kg": price,
    })

df_weekly = pd.DataFrame(weekly_rows)
df_weekly.to_csv("sample_104w_timeseries.csv", index=False)

route_rows = []
for c in CLUSTERS:
    for p in PACKHOUSES:
        dk = FC_TO_PH_DIST[(c["id"], p["id"])]
        route_rows.append({
            "Echelon": "First_Mile_Farm_to_Packhouse",
            "Origin_ID": c["id"], "Origin_Name": c["name"],
            "Destination_ID": p["id"], "Destination_Name": p["name"],
            "Distance_km": dk, "Transit_Time_Hrs": round(dk / 38.0 + 1.2, 2),
            "Ambient_Freight_INR_MT": round(180.0 + dk * 4.2, 0),
            "Reefer_Freight_INR_MT": round(260.0 + dk * 5.8, 0),
            "Ambient_Transit_Loss_Pct": round(min(0.18, 0.025 + dk * 0.00018), 4),
            "Reefer_Transit_Loss_Pct": round(min(0.06, 0.008 + dk * 0.00004), 4),
        })
for p in PACKHOUSES:
    for d in DEMAND_HUBS:
        dk = PH_TO_DH_DIST[(p["id"], d["id"])]
        route_rows.append({
            "Echelon": "Line_Haul_Packhouse_to_Hub",
            "Origin_ID": p["id"], "Origin_Name": p["name"],
            "Destination_ID": d["id"], "Destination_Name": d["name"],
            "Distance_km": dk, "Transit_Time_Hrs": round(dk / 46.0 + 1.5, 2),
            "Ambient_Freight_INR_MT": round(220.0 + dk * 3.6, 0),
            "Reefer_Freight_INR_MT": round(320.0 + dk * 5.1, 0),
            "Ambient_Transit_Loss_Pct": round(min(0.22, 0.032 + dk * 0.00016), 4),
            "Reefer_Transit_Loss_Pct": round(min(0.05, 0.009 + dk * 0.000035), 4),
        })

df_routes = pd.DataFrame(route_rows)
df_routes.to_csv("sample_network_routes.csv", index=False)

# Save Blank Input Template for Deliverable C
pd.DataFrame(columns=df_weekly.columns).to_csv("blank_input_template.csv", index=False)

print(f"Generated sample_104w_timeseries.csv ({len(df_weekly)} rows), sample_network_routes.csv ({len(df_routes)} arcs), and blank_input_template.csv.")


In [ ]:
# CELL 3: Standalone Model Validation & Test Cases (Forecasting + Goal Programming MILP)
import numpy as np
import pandas as pd
from scipy.optimize import milp, LinearConstraint, Bounds

df_ts = pd.read_csv("sample_104w_timeseries.csv")
df_rt = pd.read_csv("sample_network_routes.csv")

# 1. Forecasting Validation (Train: Weeks 1-92, Test: Weeks 93-104)
def validate_forecast(series):
    t_tr = np.arange(1, 93)
    y_tr = series[:92]
    X_tr = np.column_stack([
        np.ones(92), t_tr,
        np.sin(2 * np.pi * t_tr / 52.0), np.cos(2 * np.pi * t_tr / 52.0),
        np.sin(4 * np.pi * t_tr / 52.0), np.cos(4 * np.pi * t_tr / 52.0),
    ])
    beta, *_ = np.linalg.lstsq(X_tr, y_tr, rcond=None)
    resid = y_tr - X_tr @ beta
    phi = np.dot(resid[1:], resid[:-1]) / np.dot(resid[:-1], resid[:-1])
    t_te = np.arange(93, len(series) + 1)
    X_te = np.column_stack([
        np.ones(len(t_te)), t_te,
        np.sin(2 * np.pi * t_te / 52.0), np.cos(2 * np.pi * t_te / 52.0),
        np.sin(4 * np.pi * t_te / 52.0), np.cos(4 * np.pi * t_te / 52.0),
    ])
    preds = np.array([(X_te[h] @ beta) + (phi ** (h + 1)) * resid[-1] for h in range(len(t_te))])
    y_te = series[92:]
    mape = np.mean(np.abs((y_te - preds) / y_te)) * 100.0
    rmse = np.sqrt(np.mean((y_te - preds) ** 2))
    return mape, rmse

mape_d, rmse_d = validate_forecast(df_ts["Total_Hub_Demand_MT"].values)
mape_s, rmse_s = validate_forecast(df_ts["Total_Cluster_Supply_MT"].values)
print(f"[TEST CASE 1 - Demand Forecast W93-104] MAPE: {mape_d:.2f}% | RMSE: {rmse_d:.1f} MT")
print(f"[TEST CASE 2 - Supply Forecast W93-104] MAPE: {mape_s:.2f}% | RMSE: {rmse_s:.1f} MT")

# 2. Goal Programming + Two-Stage Stochastic MILP Solver Validation
import pulp
print(f"PuLP Version: {pulp.__version__} (available for interactive app)")


In [ ]:
# CELL 4: Write the Complete Interactive Web Application (app.py)
%%writefile app.py
import io
import math
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import pulp
from scipy.optimize import milp, LinearConstraint, Bounds
import streamlit as st

st.set_page_config(
    page_title="AgriColdChain DSS | GIM Supply Chain Analytics",
    page_icon="🥬",
    layout="wide",
    initial_sidebar_state="expanded",
)

# -------------------------------------------------------------------------
# CUSTOM STYLING
# -------------------------------------------------------------------------
st.markdown("""
<style>
    .main-header {
        background: linear-gradient(90deg, #1B4D3E 0%, #2C6E49 100%);
        padding: 1.2rem 1.5rem;
        border-radius: 10px;
        color: white;
        margin-bottom: 1rem;
    }
    .main-header h1 { margin: 0; font-size: 1.7rem; font-weight: 700; }
    .main-header p { margin: 0.25rem 0 0 0; opacity: 0.9; font-size: 0.95rem; }
    .kpi-box {
        background-color: #F4F9F6;
        border-left: 4px solid #1B4D3E;
        padding: 0.85rem 1rem;
        border-radius: 6px;
        margin-bottom: 0.5rem;
    }
    .alert-box {
        background-color: #FEF3C7;
        border-left: 4px solid #D97706;
        padding: 0.85rem 1rem;
        border-radius: 6px;
        margin: 0.5rem 0;
    }
    .rec-box {
        background-color: #ECFDF5;
        border-left: 4px solid #059669;
        padding: 0.85rem 1rem;
        border-radius: 6px;
        margin: 0.5rem 0;
    }
</style>
""", unsafe_allow_html=True)

# -------------------------------------------------------------------------
# LOAD BUILT-IN SAMPLE DATA & MASTER PARAMETERS
# -------------------------------------------------------------------------
@st.cache_data
def load_sample_data():
    df_ts = pd.read_csv("sample_104w_timeseries.csv")
    df_rt = pd.read_csv("sample_network_routes.csv")
    return df_ts, df_rt

df_sample_ts, df_sample_rt = load_sample_data()

if "active_ts" not in st.session_state:
    st.session_state["active_ts"] = df_sample_ts.copy()
if "data_source_label" not in st.session_state:
    st.session_state["data_source_label"] = "Built-in Sample Dataset (104 Weeks, Nov 2024 – Oct 2026)"

CLUSTERS_META = [
    ("FC01", "Nashik (Pimpalgaon)", "Maharashtra", "FC01_Nashik_MT"),
    ("FC02", "Narayangaon (Junner)", "Maharashtra", "FC02_Narayangaon_MT"),
    ("FC03", "Satara (Wai-Karad)", "Maharashtra", "FC03_Satara_MT"),
    ("FC04", "Sangli (Miraj-Tasgaon)", "Maharashtra", "FC04_Sangli_MT"),
    ("FC05", "Kolhapur (Shirol)", "Maharashtra", "FC05_Kolhapur_MT"),
    ("FC06", "Belagavi (Gokak)", "Karnataka", "FC06_Belagavi_MT"),
    ("FC07", "Dharwad (Hubballi)", "Karnataka", "FC07_Dharwad_MT"),
    ("FC08", "Kolar (Mulbagal)", "Karnataka", "FC08_Kolar_MT"),
]

PACKHOUSES_META = [
    {"id": "PH01", "name": "Ozarkhed Packhouse Hub (Nashik)", "cap": 850.0, "capex_l": 68.0, "fixed_wk": 145000.0, "var_mt": 620.0, "loss_red": 0.68},
    {"id": "PH02", "name": "Chakan-Narayangaon Agro Hub (Pune N)", "cap": 750.0, "capex_l": 64.0, "fixed_wk": 135000.0, "var_mt": 590.0, "loss_red": 0.66},
    {"id": "PH03", "name": "Karad-Satara Integrated Packhouse", "cap": 650.0, "capex_l": 54.0, "fixed_wk": 115000.0, "var_mt": 560.0, "loss_red": 0.65},
    {"id": "PH04", "name": "Kagal-Kolhapur Cold Chain Park", "cap": 600.0, "capex_l": 50.0, "fixed_wk": 108000.0, "var_mt": 550.0, "loss_red": 0.64},
    {"id": "PH05", "name": "Belagavi-Kittur Border Logistics Hub", "cap": 700.0, "capex_l": 56.0, "fixed_wk": 118000.0, "var_mt": 540.0, "loss_red": 0.67},
    {"id": "PH06", "name": "Hoskote-Kolar Mega Packhouse", "cap": 800.0, "capex_l": 66.0, "fixed_wk": 140000.0, "var_mt": 600.0, "loss_red": 0.68},
]

HUBS_META = [
    {"id": "DH01", "name": "Mumbai Metro DC (Vashi)", "col": "DH01_Mumbai_Demand_MT", "spot_mt": 31500.0},
    {"id": "DH02", "name": "Pune Metro DC (Marketyard)", "col": "DH02_Pune_Demand_MT", "spot_mt": 29500.0},
    {"id": "DH03", "name": "Goa Coastal Hub (Panaji/Margao)", "col": "DH03_Goa_Demand_MT", "spot_mt": 33800.0},
    {"id": "DH04", "name": "Bengaluru Metro DC (Yeshwanthpur)", "col": "DH04_Bengaluru_Demand_MT", "spot_mt": 30500.0},
]

# -------------------------------------------------------------------------
# HEADER BANNER & SIDEBAR NAVIGATION
# -------------------------------------------------------------------------
st.markdown("""
<div class="main-header">
    <h1>🥬 AgriColdChain DSS — Farm-to-Fork Cold-Chain & Stochastic Supply Planner</h1>
    <p>Goa Institute of Management | PGDM-BDA Supply Chain Analytics | Western India Horticulture Corridor (MH–GA–KA)</p>
</div>
""", unsafe_allow_html=True)

st.sidebar.title("Navigation")
page = st.sidebar.radio(
    "Select Module",
    [
        "1. Executive Dashboard (CLO 3)",
        "2. Demand & Supply Forecasting (CLO 4)",
        "3. Cold-Chain Network Optimizer (CLO 5)",
        "4. Data Upload & Templates",
        "5. Context, Strategy & Tech Outlook (CLO 1,2,6)",
        "6. User Guide & Limitations",
    ],
)
st.sidebar.markdown("---")
st.sidebar.info(f"**Active Data:**\n{st.session_state['data_source_label']}")
df_ts = st.session_state["active_ts"]

# -------------------------------------------------------------------------
# PAGE 1: EXECUTIVE DASHBOARD (DESCRIPTIVE ANALYTICS - CLO 3)
# -------------------------------------------------------------------------
if page == "1. Executive Dashboard (CLO 3)":
    st.subheader("Descriptive Analytics: Corridor Supply, Demand & Spoilage Diagnostics")
    seasons = st.multiselect("Filter by Agricultural Season", df_ts["Agri_Season"].unique(), default=list(df_ts["Agri_Season"].unique()))
    df_f = df_ts[df_ts["Agri_Season"].isin(seasons)] if seasons else df_ts

    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Avg Weekly Supply", f"{df_f['Total_Cluster_Supply_MT'].mean():,.1f} MT", f"Peak: {df_f['Total_Cluster_Supply_MT'].max():,.0f} MT")
    c2.metric("Avg Weekly Demand", f"{df_f['Total_Hub_Demand_MT'].mean():,.1f} MT", f"Peak: {df_f['Total_Hub_Demand_MT'].max():,.0f} MT")
    amb_avg = df_f["Ambient_Spoilage_Pct"].mean() * 100
    cold_avg = df_f["ColdChain_Spoilage_Pct"].mean() * 100
    c3.metric("Ambient Spoilage Rate", f"{amb_avg:.2f}%", f"+{amb_avg - cold_avg:.2f}% vs Cold Chain", delta_color="inverse")
    c4.metric("Avg Mandi Modal Price", f"₹{df_f['Mandi_Modal_Price_INR_Kg'].mean():.2f}/kg", f"Range: ₹{df_f['Mandi_Modal_Price_INR_Kg'].min():.1f}–₹{df_f['Mandi_Modal_Price_INR_Kg'].max():.1f}")

    col_left, col_right = st.columns(2)
    with col_left:
        fig_ts = go.Figure()
        fig_ts.add_trace(go.Scatter(x=df_f["Week_Start_Date"], y=df_f["Total_Cluster_Supply_MT"], name="Total Cluster Supply (MT)", line=dict(color="#1B4D3E", width=2.2)))
        fig_ts.add_trace(go.Scatter(x=df_f["Week_Start_Date"], y=df_f["Total_Hub_Demand_MT"], name="Total Urban Demand (MT)", line=dict(color="#D97706", width=2.2, dash="dot")))
        fig_ts.update_layout(title="Weekly Supply vs. Urban Hub Demand (MT)", xaxis_title="Week", yaxis_title="Metric Tonnes (MT)", height=380, legend=dict(orientation="h", y=-0.2))
        st.plotly_chart(fig_ts, use_container_width=True)

    with col_right:
        fig_sp = go.Figure()
        fig_sp.add_trace(go.Scatter(x=df_f["Week_Start_Date"], y=df_f["Ambient_Spoilage_Pct"] * 100, name="Ambient Transport Spoilage (%)", line=dict(color="#DC2626", width=2)))
        fig_sp.add_trace(go.Scatter(x=df_f["Week_Start_Date"], y=df_f["ColdChain_Spoilage_Pct"] * 100, name="Integrated Cold-Chain Spoilage (%)", line=dict(color="#059669", width=2)))
        fig_sp.update_layout(title="Post-Harvest Spoilage Trade-off: Ambient vs. Cold Chain (%)", xaxis_title="Week", yaxis_title="Spoilage (%)", height=380, legend=dict(orientation="h", y=-0.2))
        st.plotly_chart(fig_sp, use_container_width=True)

    # Cluster & Hub Breakdown
    col_b1, col_b2 = st.columns(2)
    with col_b1:
        cl_means = [{"Cluster": name, "Mean Weekly Supply (MT)": round(df_f[col].mean(), 1)} for _, name, _, col in CLUSTERS_META]
        fig_cl = px.bar(pd.DataFrame(cl_means), x="Cluster", y="Mean Weekly Supply (MT)", title="Mean Weekly Supply by Farming Cluster (MT)", color_discrete_sequence=["#2C6E49"])
        st.plotly_chart(fig_cl, use_container_width=True)
    with col_b2:
        hb_means = [{"Demand Hub": h["name"], "Mean Weekly Demand (MT)": round(df_f[h["col"]].mean(), 1)} for h in HUBS_META]
        fig_hb = px.bar(pd.DataFrame(hb_means), x="Demand Hub", y="Mean Weekly Demand (MT)", title="Mean Weekly Demand by Urban Hub (MT)", color_discrete_sequence=["#0284C7"])
        st.plotly_chart(fig_hb, use_container_width=True)

# -------------------------------------------------------------------------
# PAGE 2: FORECASTING (PREDICTIVE ANALYTICS - CLO 4)
# -------------------------------------------------------------------------
elif page == "2. Demand & Supply Forecasting (CLO 4)":
    st.subheader("Predictive Analytics: Seasonal Time-Series Forecasting & Validation")
    target_choice = st.selectbox("Select Series to Forecast", ["Total_Hub_Demand_MT", "Total_Cluster_Supply_MT", "DH01_Mumbai_Demand_MT", "DH03_Goa_Demand_MT", "Mandi_Modal_Price_INR_Kg"])
    horizon = st.slider("Forecast Horizon (Weeks Ahead)", min_value=4, max_value=16, value=8)
    train_split = st.slider("Validation Split (Training Weeks)", min_value=70, max_value=min(96, len(df_ts)-6), value=min(92, len(df_ts)-12))

    y = df_ts[target_choice].values
    n_tot = len(y)

    def run_seasonal_ar(y_arr, tr_end, h_ahead):
        t_tr = np.arange(1, tr_end + 1)
        y_tr = y_arr[:tr_end]
        X_tr = np.column_stack([
            np.ones(tr_end), t_tr,
            np.sin(2 * np.pi * t_tr / 52.0), np.cos(2 * np.pi * t_tr / 52.0),
            np.sin(4 * np.pi * t_tr / 52.0), np.cos(4 * np.pi * t_tr / 52.0),
        ])
        beta, *_ = np.linalg.lstsq(X_tr, y_tr, rcond=None)
        resid = y_tr - X_tr @ beta
        phi = np.dot(resid[1:], resid[:-1]) / max(1e-9, np.dot(resid[:-1], resid[:-1]))
        sigma = np.std(resid[1:] - phi * resid[:-1])

        t_te = np.arange(tr_end + 1, len(y_arr) + 1)
        X_te = np.column_stack([
            np.ones(len(t_te)), t_te,
            np.sin(2 * np.pi * t_te / 52.0), np.cos(2 * np.pi * t_te / 52.0),
            np.sin(4 * np.pi * t_te / 52.0), np.cos(4 * np.pi * t_te / 52.0),
        ])
        val_preds = np.array([(X_te[h] @ beta) + (phi ** (h + 1)) * resid[-1] for h in range(len(t_te))])
        val_actual = y_arr[tr_end:]
        mape = np.mean(np.abs((val_actual - val_preds) / np.maximum(1e-6, val_actual))) * 100.0
        rmse = np.sqrt(np.mean((val_actual - val_preds) ** 2))

        # Full fit for future forecast
        t_all = np.arange(1, len(y_arr) + 1)
        X_all = np.column_stack([
            np.ones(len(y_arr)), t_all,
            np.sin(2 * np.pi * t_all / 52.0), np.cos(2 * np.pi * t_all / 52.0),
            np.sin(4 * np.pi * t_all / 52.0), np.cos(4 * np.pi * t_all / 52.0),
        ])
        beta_f, *_ = np.linalg.lstsq(X_all, y_arr, rcond=None)
        res_f = y_arr - X_all @ beta_f
        phi_f = np.dot(res_f[1:], res_f[:-1]) / max(1e-9, np.dot(res_f[:-1], res_f[:-1]))
        sig_f = np.std(res_f[1:] - phi_f * res_f[:-1])

        t_fut = np.arange(len(y_arr) + 1, len(y_arr) + h_ahead + 1)
        X_fut = np.column_stack([
            np.ones(h_ahead), t_fut,
            np.sin(2 * np.pi * t_fut / 52.0), np.cos(2 * np.pi * t_fut / 52.0),
            np.sin(4 * np.pi * t_fut / 52.0), np.cos(4 * np.pi * t_fut / 52.0),
        ])
        fut_preds = np.array([(X_fut[h] @ beta_f) + (phi_f ** (h + 1)) * res_f[-1] for h in range(h_ahead)])
        fut_se = np.array([sig_f * np.sqrt(sum((phi_f ** (2 * j)) for j in range(h + 1))) for h in range(h_ahead)])
        return val_preds, mape, rmse, fut_preds, fut_preds - 1.96 * fut_se, fut_preds + 1.96 * fut_se

    val_preds, mape, rmse, fut_preds, ci_low, ci_high = run_seasonal_ar(y, train_split, horizon)

    k1, k2, k3 = st.columns(3)
    k1.metric("Out-of-Sample MAPE", f"{mape:.2f}%", "Excellent (<10%)" if mape < 10 else "Moderate")
    k2.metric("Out-of-Sample RMSE", f"{rmse:,.1f}")
    k3.metric(f"Next {horizon}-Wk Avg Forecast", f"{fut_preds.mean():,.1f}")

    fig_fc = go.Figure()
    fig_fc.add_trace(go.Scatter(x=list(range(1, n_tot + 1)), y=y, name="Actual Historical", line=dict(color="#1F2937", width=2)))
    fig_fc.add_trace(go.Scatter(x=list(range(train_split + 1, n_tot + 1)), y=val_preds, name="Validation Fit (Test)", line=dict(color="#2563EB", width=2, dash="dash")))
    fut_x = list(range(n_tot + 1, n_tot + horizon + 1))
    fig_fc.add_trace(go.Scatter(x=fut_x, y=fut_preds, name="Point Forecast", line=dict(color="#059669", width=2.5)))
    fig_fc.add_trace(go.Scatter(x=fut_x + fut_x[::-1], y=list(ci_high) + list(ci_low[::-1]), fill="toself", fillcolor="rgba(5,150,105,0.18)", line=dict(color="rgba(255,255,255,0)"), name="95% Confidence Interval"))
    fig_fc.update_layout(title=f"SARIMA / Seasonal-AR Forecast & Validation: {target_choice}", xaxis_title="Week Index", yaxis_title=target_choice, height=420)
    st.plotly_chart(fig_fc, use_container_width=True)

    fc_df = pd.DataFrame({
        "Future_Week_Index": fut_x,
        "Point_Forecast": np.round(fut_preds, 2),
        "Lower_95_CI": np.round(ci_low, 2),
        "Upper_95_CI": np.round(ci_high, 2),
    })
    st.dataframe(fc_df, use_container_width=True)
    st.download_button("Download Forecast CSV", fc_df.to_csv(index=False).encode("utf-8"), "forecast_results.csv", "text/csv")

# -------------------------------------------------------------------------
# PAGE 3: PRESCRIPTIVE OPTIMIZER (GOAL PROGRAMMING + STOCHASTIC MILP - CLO 5)
# -------------------------------------------------------------------------
elif page == "3. Cold-Chain Network Optimizer (CLO 5)":
    st.subheader("Prescriptive Analytics: Goal Programming & Two-Stage Stochastic Optimizer")
    st.caption("Adjust managerial goal weights, CapEx limits, and weather scenario probabilities, then click Run Optimizer.")

    with st.expander("Solver Parameters & Managerial Goals", expanded=True):
        g1, g2, g3 = st.columns(3)
        w_cost = g1.slider("Goal 1 Weight: Cost Control", 0.0, 1.0, 0.35, 0.05)
        w_spoil = g2.slider("Goal 2 Weight: Spoilage Reduction", 0.0, 1.0, 0.35, 0.05)
        w_short = g3.slider("Goal 3 Weight: Service Fill-Rate SLA", 0.0, 1.0, 0.30, 0.05)

        p1, p2, p3, p4 = st.columns(4)
        max_ph = p1.slider("Max Packhouses to Open", 1, 6, 4)
        cost_target_lakhs = p2.number_input("Weekly Cost Target (₹ Lakhs)", 300.0, 900.0, 560.0, 10.0)
        spoil_target_mt = p3.number_input("Weekly Spoilage Target (MT)", 20.0, 300.0, 85.0, 5.0)
        spot_cap_pct = p4.slider("Max Spot-Market Share (%)", 0, 50, 20, 5) / 100.0

        s1, s2, s3 = st.columns(3)
        p_adverse = s1.slider("Prob: Adverse Weather (0.78x Yield)", 0.0, 0.8, 0.25, 0.05)
        p_normal = s2.slider("Prob: Normal Monsoon (1.00x Yield)", 0.0, 1.0, 0.50, 0.05)
        p_bumper = max(0.0, round(1.0 - p_adverse - p_normal, 2))
        s3.metric("Prob: Bumper Glut (1.24x Yield)", f"{p_bumper:.2f}", "Auto-balanced to 1.00")

    if st.button("🚀 Run Goal Programming + Stochastic MILP Solver", type="primary"):
        if abs(p_adverse + p_normal + p_bumper - 1.0) > 0.01:
            st.error("Scenario probabilities must sum to 1.00. Please adjust Adverse and Normal probabilities.")
        else:
            scenarios = [
                {"id": "S1_Adverse", "prob": p_adverse, "yield": 0.78, "amb_sp_m": 1.35, "ref_sp_m": 1.08, "spot_m": 1.42, "dem_m": 1.03},
                {"id": "S2_Normal",  "prob": p_normal,  "yield": 1.00, "amb_sp_m": 1.00, "ref_sp_m": 1.00, "spot_m": 1.00, "dem_m": 1.00},
                {"id": "S3_Bumper",  "prob": p_bumper,  "yield": 1.24, "amb_sp_m": 1.12, "ref_sp_m": 1.02, "spot_m": 0.76, "dem_m": 0.98},
            ]
            sup_base = {cid: df_ts[col].mean() for cid, _, _, col in CLUSTERS_META}
            dem_base = {h["id"]: df_ts[h["col"]].mean() for h in HUBS_META}
            rt_map = {(r["Origin_ID"], r["Destination_ID"]): r for _, r in df_sample_rt.iterrows()}
            ph_ids = [p["id"] for p in PACKHOUSES_META]
            ph_dict = {p["id"]: p for p in PACKHOUSES_META}
            cl_ids = [c[0] for c in CLUSTERS_META]
            hb_ids = [h["id"] for h in HUBS_META]
            hb_dict = {h["id"]: h for h in HUBS_META}
            modes = ["AMB", "REF"]

            # Build MILP using SciPy HiGHS (fast, deterministic)
            idx_y = {j: i for i, j in enumerate(ph_ids)}
            offset = len(ph_ids)
            idx_x, idx_z, idx_spot, idx_short = {}, {}, {}, {}
            for s in scenarios:
                for i in cl_ids:
                    for j in ph_ids:
                        for m in modes:
                            idx_x[(i, j, m, s["id"])] = offset; offset += 1
                for j in ph_ids:
                    for k in hb_ids:
                        for m in modes:
                            idx_z[(j, k, m, s["id"])] = offset; offset += 1
                for k in hb_ids:
                    idx_spot[(k, s["id"])] = offset; offset += 1
                    idx_short[(k, s["id"])] = offset; offset += 1

            idx_dcost, idx_dspoil, idx_dshort = offset, offset + 1, offset + 2
            n_vars = offset + 3

            c_obj = np.zeros(n_vars)
            c_obj[idx_dcost] = w_cost / (cost_target_lakhs * 100000.0)
            c_obj[idx_dspoil] = w_spoil / max(1.0, spoil_target_mt)
            c_obj[idx_dshort] = w_short / 120.0
            for j in ph_ids:
                c_obj[idx_y[j]] += 1e-9 * (ph_dict[j]["fixed_wk"] + ph_dict[j]["capex_l"] * 100000.0 / 52.0)

            rows_A, lb_A, ub_A = [], [], []
            def add_con(coeffs, lb, ub):
                row = np.zeros(n_vars)
                for idx, val in coeffs.items():
                    row[idx] += val
                rows_A.append(row); lb_A.append(lb); ub_A.append(ub)

            for s in scenarios:
                for i in cl_ids:
                    add_con({idx_x[(i, j, m, s["id"])]: 1.0 for j in ph_ids for m in modes}, 0.0, sup_base[i] * s["yield"])
                for j in ph_ids:
                    coeffs = {idx_x[(i, j, m, s["id"])]: 1.0 for i in cl_ids for m in modes}
                    coeffs[idx_y[j]] = -ph_dict[j]["cap"]
                    add_con(coeffs, -np.inf, 0.0)
                    # Flow balance
                    fb = {}
                    for i in cl_ids:
                        r = rt_map[(i, j)]
                        fb[idx_x[(i, j, "AMB", s["id"])]] = 1.0 - min(0.25, r["Ambient_Transit_Loss_Pct"] * s["amb_sp_m"])
                        fb[idx_x[(i, j, "REF", s["id"])]] = 1.0 - min(0.10, r["Reefer_Transit_Loss_Pct"] * s["ref_sp_m"] * (1 - 0.3 * ph_dict[j]["loss_red"]))
                    for k in hb_ids:
                        for m in modes:
                            fb[idx_z[(j, k, m, s["id"])]] = -1.0
                    add_con(fb, 0.0, 0.0)
                for k in hb_ids:
                    dc = {}
                    for j in ph_ids:
                        r = rt_map[(j, k)]
                        dc[idx_z[(j, k, "AMB", s["id"])]] = 1.0 - min(0.25, r["Ambient_Transit_Loss_Pct"] * s["amb_sp_m"] * (1 - 0.5 * ph_dict[j]["loss_red"]))
                        dc[idx_z[(j, k, "REF", s["id"])]] = 1.0 - min(0.08, r["Reefer_Transit_Loss_Pct"] * s["ref_sp_m"] * (1 - ph_dict[j]["loss_red"]))
                    dc[idx_spot[(k, s["id"])]] = 1.0
                    dc[idx_short[(k, s["id"])]] = 1.0
                    dem_sk = dem_base[k] * s["dem_m"]
                    add_con(dc, dem_sk, dem_sk)
                    add_con({idx_spot[(k, s["id"])]: 1.0}, 0.0, spot_cap_pct * dem_sk)

            add_con({idx_y[j]: 1.0 for j in ph_ids}, 1.0, float(max_ph))

            cost_coeffs = {idx_y[j]: ph_dict[j]["fixed_wk"] + ph_dict[j]["capex_l"] * 100000.0 / 52.0 for j in ph_ids}
            spoil_coeffs = {}
            for s in scenarios:
                p = s["prob"]
                if p <= 0:
                    continue
                for i in cl_ids:
                    for j in ph_ids:
                        r = rt_map[(i, j)]
                        cost_coeffs[idx_x[(i, j, "AMB", s["id"])]] = p * (18000.0 + r["Ambient_Freight_INR_MT"] + ph_dict[j]["var_mt"])
                        cost_coeffs[idx_x[(i, j, "REF", s["id"])]] = p * (18000.0 + r["Reefer_Freight_INR_MT"] + ph_dict[j]["var_mt"])
                        spoil_coeffs[idx_x[(i, j, "AMB", s["id"])]] = p * min(0.25, r["Ambient_Transit_Loss_Pct"] * s["amb_sp_m"])
                        spoil_coeffs[idx_x[(i, j, "REF", s["id"])]] = p * min(0.10, r["Reefer_Transit_Loss_Pct"] * s["ref_sp_m"] * (1 - 0.3 * ph_dict[j]["loss_red"]))
                for j in ph_ids:
                    for k in hb_ids:
                        r = rt_map[(j, k)]
                        cost_coeffs[idx_z[(j, k, "AMB", s["id"])]] = p * r["Ambient_Freight_INR_MT"]
                        cost_coeffs[idx_z[(j, k, "REF", s["id"])]] = p * r["Reefer_Freight_INR_MT"]
                        spoil_coeffs[idx_z[(j, k, "AMB", s["id"])]] = p * min(0.25, r["Ambient_Transit_Loss_Pct"] * s["amb_sp_m"] * (1 - 0.5 * ph_dict[j]["loss_red"]))
                        spoil_coeffs[idx_z[(j, k, "REF", s["id"])]] = p * min(0.08, r["Reefer_Transit_Loss_Pct"] * s["ref_sp_m"] * (1 - ph_dict[j]["loss_red"]))
                for k in hb_ids:
                    cost_coeffs[idx_spot[(k, s["id"])]] = p * hb_dict[k]["spot_mt"] * s["spot_m"]
                    cost_coeffs[idx_short[(k, s["id"])]] = p * 42000.0

            cost_g = dict(cost_coeffs); cost_g[idx_dcost] = -1.0
            add_con(cost_g, -np.inf, cost_target_lakhs * 100000.0)
            spoil_g = dict(spoil_coeffs); spoil_g[idx_dspoil] = -1.0
            add_con(spoil_g, -np.inf, spoil_target_mt)
            short_coeffs = {idx_short[(k, s["id"])]: s["prob"] for s in scenarios for k in hb_ids}
            short_g = dict(short_coeffs); short_g[idx_dshort] = -1.0
            add_con(short_g, -np.inf, 0.0)

            integrality = np.zeros(n_vars); integrality[:len(ph_ids)] = 1
            ub_v = np.full(n_vars, np.inf); ub_v[:len(ph_ids)] = 1.0
            res = milp(c=c_obj, integrality=integrality, bounds=Bounds(np.zeros(n_vars), ub_v), constraints=LinearConstraint(np.array(rows_A), lb_A, ub_A))

            if res.success:
                opened = [ph_dict[ph_ids[i]]["name"] for i in range(len(ph_ids)) if res.x[i] > 0.5]
                exp_cost_l = sum(res.x[idx] * val for idx, val in cost_coeffs.items()) / 100000.0
                exp_spoil_mt = sum(res.x[idx] * val for idx, val in spoil_coeffs.items())
                exp_short_mt = sum(res.x[idx] * val for idx, val in short_coeffs.items())
                tot_dem = sum(dem_base.values())
                fill_rate = (1.0 - exp_short_mt / tot_dem) * 100.0
                spoil_pct = (exp_spoil_mt / tot_dem) * 100.0

                m1, m2, m3, m4 = st.columns(4)
                m1.metric("Packhouses Opened", f"{len(opened)} / {max_ph}")
                m2.metric("Expected Weekly Cost", f"₹{exp_cost_l:,.2f} L", f"Target: ₹{cost_target_lakhs:.0f} L")
                m3.metric("Expected Spoilage", f"{spoil_pct:.2f}% ({exp_spoil_mt:.1f} MT)", f"-{11.61 - spoil_pct:.2f}% vs NABCONS Avg")
                m4.metric("Expected Fill Rate SLA", f"{fill_rate:.2f}%", f"Shortfall: {exp_short_mt:.1f} MT")

                st.success(f"**Optimal Packhouse Locations Selected:** {', '.join(opened)}")

                # Managerial Recommendations & Alerts
                if exp_short_mt > 50.0:
                    st.markdown(f"<div class='alert-box'><b>⚠️ SLA Capacity Alert:</b> Expected shortfall is {exp_short_mt:.1f} MT/week. Consider increasing Max Packhouses or relaxing the spot-market cap.</div>", unsafe_allow_html=True)
                else:
                    st.markdown(f"<div class='rec-box'><b>✅ Managerial Recommendation:</b> Opening {len(opened)} strategic packhouses achieves a {fill_rate:.1f}% fill rate while cutting post-harvest spoilage from the 11.61% NABCONS ambient baseline down to {spoil_pct:.2f}%.</div>", unsafe_allow_html=True)

                # Packhouse Utilization Table (Normal Scenario)
                ph_rows = []
                for j in ph_ids:
                    is_open = res.x[idx_y[j]] > 0.5
                    flow_n = sum(res.x[idx_x[(i, j, m, "S2_Normal")]] for i in cl_ids for m in modes) if p_normal > 0 else 0.0
                    ph_rows.append({
                        "Packhouse_ID": j,
                        "Facility_Name": ph_dict[j]["name"],
                        "Status": "OPEN" if is_open else "CLOSED",
                        "Capacity_MT_Wk": ph_dict[j]["cap"],
                        "Normal_Scenario_Flow_MT": round(flow_n, 1),
                        "Utilization_Pct": round((flow_n / ph_dict[j]["cap"]) * 100, 1) if is_open else 0.0,
                    })
                df_ph_out = pd.DataFrame(ph_rows)
                st.dataframe(df_ph_out, use_container_width=True)
                fig_util = px.bar(df_ph_out[df_ph_out["Status"] == "OPEN"], x="Facility_Name", y="Utilization_Pct", title="Opened Packhouse Capacity Utilization (% - Normal Scenario)", color_discrete_sequence=["#1B4D3E"])
                st.plotly_chart(fig_util, use_container_width=True)

                st.download_button("Download Optimization Results (CSV)", df_ph_out.to_csv(index=False).encode("utf-8"), "optimal_packhouse_decisions.csv", "text/csv")
            else:
                st.error(f"Optimization failed: {res.message}")

# -------------------------------------------------------------------------
# PAGE 4: DATA UPLOAD, VALIDATION & TEMPLATES
# -------------------------------------------------------------------------
elif page == "4. Data Upload & Templates":
    st.subheader("Data Input Management, Validation & Blank Template")
    col_u1, col_u2 = st.columns(2)
    with col_u1:
        st.download_button(
            "📥 Download Blank Input Template (CSV)",
            pd.DataFrame(columns=df_sample_ts.columns).to_csv(index=False).encode("utf-8"),
            "blank_input_template.csv",
            "text/csv",
        )
    with col_u2:
        if st.button("🔄 Reset to Built-In 104-Week Sample Dataset"):
            st.session_state["active_ts"] = df_sample_ts.copy()
            st.session_state["data_source_label"] = "Built-in Sample Dataset (104 Weeks)"
            st.success("Reset to built-in 104-week sample dataset.")

    uploaded = st.file_uploader("Upload Custom Weekly Time-Series CSV", type=["csv"])
    if uploaded is not None:
        try:
            df_up = pd.read_csv(uploaded)
            req_cols = set(df_sample_ts.columns)
            missing_cols = req_cols - set(df_up.columns)
            if missing_cols:
                st.error(f"Schema Validation Error: Missing required columns: {sorted(missing_cols)}")
            elif len(df_up) < 12:
                st.error("Range Validation Error: Dataset must contain at least 12 weekly rows for analysis.")
            elif (df_up["Total_Cluster_Supply_MT"] < 0).any() or (df_up["Total_Hub_Demand_MT"] < 0).any():
                st.error("Value Validation Error: Supply and Demand quantities must be non-negative.")
            else:
                st.session_state["active_ts"] = df_up
                st.session_state["data_source_label"] = f"Custom Upload ({uploaded.name}, {len(df_up)} rows)"
                st.success(f"Validated and loaded {len(df_up)} rows from {uploaded.name}!")
        except Exception as e:
            st.error(f"File Parsing Error: {e}")

    st.dataframe(st.session_state["active_ts"].head(15), use_container_width=True)

# -------------------------------------------------------------------------
# PAGE 5: CONTEXT, PROBLEM FRAMING & TECH OUTLOOK (CLO 1, 2, 6)
# -------------------------------------------------------------------------
elif page == "5. Context, Strategy & Tech Outlook (CLO 1,2,6)":
    st.subheader("Supply Chain Context, Problem Framing & Technology Outlook")
    tab_c1, tab_c2, tab_c6 = st.tabs(["CLO 1: Context & Strategy", "CLO 2: Problem Framing", "CLO 6: Technology Outlook"])

    with tab_c1:
        st.markdown("""
        #### Supply Chain Context & Strategic Fit
        - **Key Players:** Smallholder farmers & FPOs (8 horticulture clusters across Nashik, Pune, Satara, Sangli, Kolhapur, Belagavi, Dharwad, Kolar), Integrated Packhouses/Pre-Cooling Hubs, Reefer/Ambient Fleet Operators, and 4 Urban Demand Hubs (Mumbai, Pune, Goa, Bengaluru).
        - **Flows:** *Material* (fresh produce from farmgate to urban DCs), *Information* (weekly Agmarknet arrivals, weather alerts, DC fill-rate SLAs), and *Money* (farmgate procurement payouts, packhouse CapEx/OpEx, wholesale revenue, shortfall penalties).
        - **Strategic Fit:** Competes on **Responsive, Low-Wastage Fulfilment** (Chopra's Strategic Fit). Investing in pre-cooling packhouses and reefer dispatch trades higher fixed/freight cost for a ~9% reduction in post-harvest spoilage (from 11.61% NABCONS ambient baseline to <2.5%).
        """)
        st.markdown("#### Systems View (Causal Loop Diagram) & Business Architecture")
        st.markdown("""
        - **Reinforcing Spoilage Loop:** Higher Ambient Temp → Faster Respiration/Decay → Higher Transit Spoilage → Lower Net Supply → Higher Spot Prices & SLA Penalties.
        - **Balancing Cold-Chain Loop:** Packhouse Pre-Cooling + Reefer Allocation → Lower Field Heat (42°C → 4°C) → Extended Shelf Life → Higher Fill Rate & Margin Stability.
        - **Business Architecture:** Historical Mandi/Weather Data + Forecast Engine (SARIMA/Seasonal-AR) → Goal Programming & Stochastic MILP Engine (PuLP/HiGHS) → Manager Decision Dashboard (Facility Open/Close & Reefer Routing).
        """)

    with tab_c2:
        st.markdown("""
        #### Quantitative Problem Framing (CLO 2)
        - **Decision Variables:** Binary facility location $y_j \in \{0,1\}$ (open packhouse $j \in \{1..6\}$); continuous first-mile flows $x_{ijms} \ge 0$ and line-haul flows $z_{jkms} \ge 0$ across modes $m \in \{\text{AMB}, \text{REF}\}$ and weather scenarios $s \in \{\text{Adverse}, \text{Normal}, \text{Bumper}\}$; spot purchases $u_{ks}$ and shortfall $v_{ks}$; goal deviations $d_1^+, d_2^+, d_3^+$.
        - **Objective Function (Weighted Goal Programming):** Minimize normalized deviations $\frac{w_1}{T_{\text{cost}}} d_1^+ + \frac{w_2}{T_{\text{spoil}}} d_2^+ + \frac{w_3}{T_{\text{short}}} d_3^+$ across expected weekly cost, spoilage MT, and SLA shortfall.
        - **Constraints:** Cluster supply limits, packhouse capacity & linking ($\sum x \le C_j y_j$), spoilage-adjusted flow conservation at packhouses, demand satisfaction at hubs, max packhouses budget cap, and spot-market traceability cap.
        """)

    with tab_c6:
        st.markdown("""
        #### Technology Outlook (~1 Page Equivalent, CLO 6)
        1. **IoT Real-Time Cold-Chain Telemetry & Ethylene/Temperature Sensors:**
           - *Benefits:* Continuous temperature/humidity logging inside reefer trucks and solar cold rooms enables dynamic shelf-life estimation and Dynamic FEFO (First-Expired-First-Out) rerouting.
           - *Limitations & Challenges:* Rural cellular dead-zones along state highways, sensor calibration drift, and upfront hardware cost for smallholder FPOs.
           - *Ethical Issues:* Data ownership disputes (who owns farmgate quality logs—the aggregator or the FPO?) and algorithmic price-penalization of farmers for transit delays outside their control.
        2. **Agentic AI & Digital Twin Supply Network Simulation:**
           - *Benefits:* A live Digital Twin mirrors packhouse queues and highway congestion, while Agentic AI monitors IMD heatwave alerts, automatically triggers re-forecasts, and drafts reefer dispatch changes for manager approval.
           - *Limitations & Challenges:* High integration effort across fragmented mandi/ERP systems and risk of hallucinated or overconfident automated dispatch during unprecedented climate shocks.
           - *Ethical Issues:* Human-in-the-loop accountability (who bears liability if an autonomous agent misroutes a perishable load?) and transparency of automated vendor/FPO selection.
        """)

# -------------------------------------------------------------------------
# PAGE 6: USER GUIDE & LIMITATIONS
# -------------------------------------------------------------------------
else:
    st.subheader("User Guide, Assumptions & Model Limitations")
    st.markdown("""
    #### How to Use This App
    1. **Explore Historical Trends (Tab 1):** Filter by agricultural season (`Rabi`, `Zaid`, `Kharif`) to inspect supply/demand and ambient vs. cold-chain spoilage.
    2. **Run Forecasts (Tab 2):** Choose a series, adjust the forecast horizon (4–16 weeks), inspect out-of-sample MAPE/RMSE, and download the forecast CSV.
    3. **Optimize the Network (Tab 3):** Set your Goal Programming weights (Cost, Spoilage, SLA), CapEx limit (Max Packhouses), and weather probabilities, then click **Run Optimizer** to get actionable packhouse & utilization recommendations.
    4. **Test Custom Data (Tab 4):** Download the blank CSV template, enter your own weekly numbers, and upload it with automatic schema/range validation.

    #### Key Assumptions & Limitations
    - **Disclosed Synthetic Data:** The 104-week dataset is synthetically generated (`seed=42`) and calibrated to NABCONS (2022) tomato loss benchmarks (11.61%) and NCCD cold-chain parameters.
    - **Static Linear Freight & Spoilage:** The MILP uses deterministic arc-level spoilage percentages per scenario rather than continuous nonlinear Arrhenius thermal decay.
    - **Weekly Aggregation:** Decisions are modeled at a weekly cadence; intra-week daily dispatch scheduling is outside the current model scope.
    """)


In [ ]:
# CELL 5: Launch Locally Hosted Decision-Support Web App
import socket
import subprocess
import sys
import time

def find_free_port(start_port=8501):
    port = start_port
    while port < start_port + 20:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(("localhost", port)) != 0:
                return port
        port += 1
    return start_port

port = find_free_port(8501)
cmd = [
    sys.executable, "-m", "streamlit", "run", "app.py",
    "--server.port", str(port),
    "--server.headless", "true",
    "--browser.gatherUsageStats", "false",
]

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(3)
print(f"==================================================================")
print(f"AgriColdChain Decision-Support App is LIVE locally!")
print(f"Open in your browser: http://localhost:{port}")
print(f"To stop the server later, run: proc.terminate()")
print(f"==================================================================")
